# Laporan Tugas: Pembangunan Model Deteksi Penyakit Ginjal Kronik (CKD)

**Dataset**: `dataset/penyakit_ginjal_kronik.csv` (400 pasien, 26 kolom mentah).

**Tugas**: **Binary classification** deteksi Chronic Kidney Disease.

| Target | Arti |
|--------|------|
| `klasifikasi = ckd` | Terdeteksi penyakit ginjal kronik (label 1) |
| `klasifikasi = notckd` | Tidak terdeteksi (label 0) |

**Base method utama**: **W-KNN (Weighted K-Nearest Neighbors)**.
Lima algoritma diuji untuk memastikan pemilihan model didukung
bukti empiris (F1, ROC AUC, stabilitas CV).


## 🗺️ Pemetaan Step ke Tahap Pipeline ML

```
Pengumpulan Data
        ↓
Pemahaman Data (EDA)
        ↓
Preprocessing Data
        ↓
Feature Engineering / Seleksi Fitur
        ↓
Split Data (Train-Test)
        ↓
Training Model
        ↓
Evaluasi Model
        ↓
Tuning Model
```

| Step | Tahap Pipeline | Keterangan |
|------|----------------|------------|
| Step 1 | _Setup_ | Import library |
| Step 2 | 📥 Pengumpulan Data | Load `penyakit_ginjal_kronik.csv` |
| Step 3 | 🔍 EDA + 🧹 Preprocessing | Discovery-driven cleaning |
| Step 4 | 🔍 Pemahaman Data (EDA) | Distribusi & korelasi |
| Step 5 | 🛠️ Feature Engineering + 🧹 Encoding | LabelEncode binary cols |
| Step 6 | ✂️ Split Data | 80/20 stratified |
| Step 7 | 🧹 Preprocessing (scaling) + 🤖 Training | 60 kombinasi |
| Step 8 | 📊 Evaluasi Model | Ringkasan eksperimen |
| Step 9 | 📊 Evaluasi Model (CV) | 5-fold cross validation |
| Step 10 | 🎛️ Tuning Model | RandomizedSearchCV pada W-KNN |
| Step 11 | 📊 Evaluasi mendalam | CM, ROC, importance proxy |
| Step 12 | 📊 Head-to-head | Perbandingan final semua kandidat |


## STEP 1 - Import Library

> 🏷️ **Fase Pipeline:** _Setup_

### Tujuan
Menyiapkan semua library: pandas/numpy untuk data, seaborn/matplotlib
untuk visualisasi, scikit-learn untuk model & evaluasi, imbalanced-learn
untuk resampling, dan XGBoost sebagai pembanding.


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
print('Semua library berhasil diimport!')


## STEP 2 - Memuat Dataset

> 🏷️ **Fase Pipeline:** 📥 Pengumpulan Data

Dataset CKD klasik berisi 400 pasien dengan 25 fitur klinis (di luar `id`)
dan target `klasifikasi`. Kolom mencakup data urin, darah, dan riwayat
penyakit penyerta.


In [ ]:
df = pd.read_csv('dataset/penyakit_ginjal_kronik.csv')
print('Shape:', df.shape)
df.head()


In [ ]:
df.info()


## STEP 3 - Pembersihan Data: Investigasi → Pembuktian → Aksi

> 🏷️ **Fase Pipeline:** 🔍 EDA + 🧹 Preprocessing  (lihat tabel sub-tahap di atas)

### Filosofi
Untuk setiap potensi masalah, urutannya: **cek → buktikan → tetapkan
aturan → aksi → verifikasi**. Dataset CKD ini banyak masalah:
whitespace tersembunyi, kolom numerik tersimpan sebagai string, label
target tidak konsisten (`'ckd'` vs `'ckd\t'`), missing values masif,
dan duplikat nama kolom (`seldarahmerah` ada kategori dan numerik).


### 3.1 Cek Missing Values

**Pertanyaan:** apakah ada nilai kosong dan seberapa parah?


In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
print('Top 10 kolom dengan missing values terbanyak:')
print(miss_df.head(10))
print(f'\nTotal missing values: {missing.sum()}')


**Kesimpulan 3.1:** missing values sangat masif (sampai 38% di kolom
`seldarahmerah`). **Imputasi wajib dilakukan**, drop tidak feasible
(akan kehilangan terlalu banyak data).


### 3.2 Cek Inkonsistensi String (whitespace tersembunyi)

**Pertanyaan:** apakah nilai kategorikal punya whitespace yang membuat
kategori sama dianggap berbeda?


In [ ]:
# Cek nilai unique kolom-kolom yang biasanya punya masalah whitespace
problem_cols = ['diabetes', 'cad', 'klasifikasi']
for c in problem_cols:
    uniq = df[c].dropna().unique()
    print(f'{c:15}: {[repr(x) for x in uniq]}')


**Bukti:** kolom `diabetes` punya nilai `'yes'`, `' yes'`, `'\tno'`,
`'\tyes'`. Kolom target `klasifikasi` punya `'ckd'` dan `'ckd\t'`.
Tanpa pembersihan, model akan menganggap mereka kategori berbeda.

**Aksi:** strip whitespace dari semua kolom string + lowercase.


In [ ]:
def _strip(x):
    if isinstance(x, str):
        return x.strip().lower()
    return x

# Pandas modern: pakai .map per kolom object (applymap deprecated)
for c in df.columns:
    if df[c].dtype == object:
        df[c] = df[c].map(_strip)

# Verifikasi
for c in problem_cols:
    uniq = df[c].dropna().unique()
    print(f'{c:15}: {list(uniq)}')


### 3.3 Coerce Kolom Numerik & Rename Kolom Duplikat

**Pertanyaan:** apakah ada kolom numerik yang tersimpan sebagai string?


In [ ]:
print('Dtype awal kolom yang seharusnya numerik:')
print(df[['MCV', 'seldarahputih', 'seldarahmerah.1']].dtypes)
print('\nContoh nilai (perhatikan ada whitespace/?):')
print(df[['MCV', 'seldarahputih', 'seldarahmerah.1']].head())


**Bukti:** kolom `MCV`, `seldarahputih`, `seldarahmerah.1` bertipe
`object` padahal isinya angka. Karena `applymap(_strip)` di atas sudah
strip whitespace, sekarang tinggal `pd.to_numeric` dengan `errors='coerce'`
untuk konversi paksa.

Selain itu kolom `seldarahmerah` (kategori `'normal'/'abnormal'`) bentrok
nama dengan `seldarahmerah.1` (numerik). Kita rename agar jelas.


In [ ]:
df = df.rename(columns={
    'seldarahmerah': 'seldarahmerah_kat',
    'seldarahmerah.1': 'seldarahmerah_count',
    'MCV': 'mcv',
})

NUMERIC_COLS = [
    'umur', 'tekanandarah', 'gravitas', 'albumin', 'sugar',
    'gds', 'ureum', 'kreatinin', 'natrium', 'kalium',
    'hemoglobin', 'mcv', 'seldarahputih', 'seldarahmerah_count',
]
for c in NUMERIC_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df.dtypes


### 3.4 Drop kolom `id` dan Mapping Target ke 0/1

**Aksi:** kolom `id` tidak prediktif, drop. Target `klasifikasi`
dimapping: `ckd` → 1, `notckd` → 0.


In [ ]:
if 'id' in df.columns:
    df = df.drop(columns=['id'])

target_map = {'ckd': 1, 'notckd': 0}
df['target'] = df['klasifikasi'].map(target_map)
df = df.dropna(subset=['target']).reset_index(drop=True)
df['target'] = df['target'].astype(int)
df = df.drop(columns=['klasifikasi'])

print('Distribusi target:')
print(df['target'].value_counts())
print('\nProporsi:')
print(df['target'].value_counts(normalize=True).round(3))


### 3.5 Imputasi Missing Values

**Strategi:**
- Numerik: **median** (robust terhadap outlier).
- Kategorikal: **most frequent** (modus).


In [ ]:
CATEGORICAL_COLS = [
    'seldarahmerah_kat', 'pussel', 'puscell', 'bakteri',
    'hipertensi', 'diabetes', 'cad', 'nafsumakan', 'edema', 'anemia',
]

# Imputasi numerik
imp_num = SimpleImputer(strategy='median')
df[NUMERIC_COLS] = imp_num.fit_transform(df[NUMERIC_COLS])

# Imputasi kategorikal
imp_cat = SimpleImputer(strategy='most_frequent')
df[CATEGORICAL_COLS] = imp_cat.fit_transform(df[CATEGORICAL_COLS])

print('Missing values setelah imputasi:')
print(df.isnull().sum().sum())


### 3.6 Verifikasi Pembersihan


In [ ]:
print('Shape final:', df.shape)
print('\nDistribusi target:')
print(df['target'].value_counts())
print('\nDtypes (semua harus numerik atau object kategorikal):')
print(df.dtypes)


## STEP 4 - Eksplorasi Data (EDA)

> 🏷️ **Fase Pipeline:** 🔍 Pemahaman Data (EDA)

Lihat distribusi target, distribusi fitur penting, dan korelasi
fitur numerik dengan target.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.countplot(data=df, x='target', ax=axes[0])
axes[0].set_title('Distribusi Target')
axes[0].set_xticklabels(['Tidak CKD (0)', 'CKD (1)'])

sns.histplot(data=df, x='hemoglobin', hue='target', bins=20,
             multiple='stack', ax=axes[1])
axes[1].set_title('Hemoglobin per kelas')

sns.histplot(data=df, x='kreatinin', hue='target', bins=20,
             multiple='stack', ax=axes[2])
axes[2].set_title('Kreatinin per kelas')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=df, x='target', y='kreatinin', ax=axes[0])
axes[0].set_xticklabels(['Tidak CKD', 'CKD'])
axes[0].set_title('Kreatinin vs Target')

sns.boxplot(data=df, x='target', y='hemoglobin', ax=axes[1])
axes[1].set_xticklabels(['Tidak CKD', 'CKD'])
axes[1].set_title('Hemoglobin vs Target')
plt.tight_layout()
plt.show()


## STEP 5 - Encoding Kolom Kategorikal

> 🏷️ **Fase Pipeline:** 🛠️ Feature Engineering + 🧹 Preprocessing (encoding)

Semua kolom kategorikal di dataset ini sudah binary (`yes`/`no`,
`normal`/`abnormal`, `present`/`notpresent`, `good`/`poor`), jadi
**LabelEncoder** sudah cukup tanpa one-hot encoding.

Tidak ada feature engineering numerik tambahan karena fitur-fitur
sudah berupa pengukuran klinis langsung dengan makna yang jelas.


In [ ]:
df_enc = df.copy()
encoders = {}
for c in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_enc[c] = le.fit_transform(df_enc[c].astype(str))
    encoders[c] = le
    print(f'{c:25}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

df_enc.head()


In [ ]:
# Korelasi semua fitur dengan target
corr = df_enc.corr(numeric_only=True)['target'].drop('target').sort_values()

plt.figure(figsize=(7, 7))
corr.plot(kind='barh', color='steelblue')
plt.title('Korelasi setiap fitur dengan target CKD')
plt.tight_layout()
plt.show()
corr


## STEP 6 - Pemisahan Fitur, Target, dan Train/Test Split

> 🏷️ **Fase Pipeline:** ✂️ Split Data (Train-Test)

- 80% train / 20% test, **stratified** agar proporsi target tetap.


In [ ]:
feature_cols = NUMERIC_COLS + CATEGORICAL_COLS
target_col = 'target'

X = df_enc[feature_cols]
y = df_enc[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Distribusi train:')
print(pd.Series(y_train).value_counts(normalize=True).round(3))
print('Distribusi test:')
print(pd.Series(y_test).value_counts(normalize=True).round(3))


## STEP 7 - Eksperimen Sistematis: Scaler × Resampler × Model

> 🏷️ **Fase Pipeline:** 🧹 Preprocessing (scaling/resampling) + 🤖 Training Model

Eksperimen 60 kombinasi (3 scaler × 4 resampler × 5 model).

**W-KNN** sangat sensitif pada scaling karena bekerja di ruang Euclidean,
jadi pemilihan scaler kritis untuk performa.


In [ ]:
SCALERS = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler(),
}

RESAMPLERS = {
    'None': None,
    'SMOTE': SMOTE(random_state=RANDOM_STATE),
    'ADASYN': ADASYN(random_state=RANDOM_STATE),
    'RandomUnderSampler': RandomUnderSampler(random_state=RANDOM_STATE),
}

MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Extra Trees': ExtraTreesClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'W-KNN': KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                             random_state=RANDOM_STATE, use_label_encoder=False,
                             eval_metric='logloss', n_jobs=-1),
}
print('Total kombinasi:', len(SCALERS) * len(RESAMPLERS) * len(MODELS))


In [ ]:
def evaluate(model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    try:
        y_proba = model.predict_proba(X_te)[:, 1]
        roc = roc_auc_score(y_te, y_proba)
    except Exception:
        roc = None
    return {
        'accuracy': accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred, zero_division=0),
        'recall': recall_score(y_te, y_pred, zero_division=0),
        'f1': f1_score(y_te, y_pred, zero_division=0),
        'roc_auc': roc,
    }


def _safe_resample(resampler, X_tr, y_tr):
    if resampler is None:
        return X_tr, y_tr.copy(), False
    try:
        X_r, y_r = clone(resampler).fit_resample(X_tr, y_tr)
        return X_r, y_r, False
    except (ValueError, RuntimeError) as exc:
        print(f"   [WARN] Resampler gagal -> fallback ke data asli: {exc}")
        return X_tr, y_tr.copy(), True


results = []
for sc_name, scaler in SCALERS.items():
    X_tr_s = scaler.fit_transform(X_train)
    X_te_s = scaler.transform(X_test)
    for rs_name, resampler in RESAMPLERS.items():
        X_tr_r, y_tr_r, fallback = _safe_resample(resampler, X_tr_s, y_train)
        for mdl_name, mdl in MODELS.items():
            m = clone(mdl)
            metrics = evaluate(m, X_tr_r, X_te_s, y_tr_r, y_test)
            results.append({'Scaler': sc_name, 'Resampler': rs_name,
                            'Model': mdl_name, 'fallback_no_resample': fallback,
                            **metrics})
            tag = ' (fallback)' if fallback else ''
            print(f"[{sc_name:>14} | {rs_name:>18}{tag:<11} | {mdl_name:>20}] "
                  f"F1={metrics['f1']:.4f}  AUC={metrics['roc_auc']:.4f}")

results_df = pd.DataFrame(results)
print('\nTotal hasil:', len(results_df))
results_df.head()


## STEP 8 - Ringkasan Eksperimen

> 🏷️ **Fase Pipeline:** 📊 Evaluasi Model

Top kombinasi global biasanya didominasi 1 model. Untuk fairness, kita
juga tampilkan **best per model** dan pivot Model × Scaler / Resampler.


In [ ]:
results_sorted = results_df.sort_values('f1', ascending=False).reset_index(drop=True)
print(f'Total kombinasi: {len(results_sorted)}')
print('\n=== Top-10 kombinasi global ===')
results_sorted.head(10).round(4)


In [ ]:
print('Distribusi model di Top-10:')
print(results_sorted.head(10)['Model'].value_counts().to_string())
print('\nDistribusi model di Top-20:')
print(results_sorted.head(20)['Model'].value_counts().to_string())


In [ ]:
best_per_model = (results_sorted
                  .sort_values('f1', ascending=False)
                  .groupby('Model', as_index=False)
                  .first()
                  .sort_values('f1', ascending=False)
                  .reset_index(drop=True))
print('=== Konfigurasi terbaik per model ===')
best_per_model[['Model', 'Scaler', 'Resampler',
                'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].round(4)


In [ ]:
pivot_scaler = (results_df.pivot_table(index='Model', columns='Scaler',
                                       values='f1', aggfunc='mean').round(4))
pivot_resampler = (results_df.pivot_table(index='Model', columns='Resampler',
                                          values='f1', aggfunc='mean').round(4))
print('=== Rata-rata F1 per (Model, Scaler) ===')
print(pivot_scaler)
print('\n=== Rata-rata F1 per (Model, Resampler) ===')
print(pivot_resampler)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.heatmap(pivot_scaler, annot=True, fmt='.4f', cmap='YlGnBu', ax=axes[0])
axes[0].set_title('Rata-rata F1 per Model × Scaler')
sns.heatmap(pivot_resampler, annot=True, fmt='.4f', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Rata-rata F1 per Model × Resampler')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=best_per_model, x='Model', y='f1', palette='viridis', ax=axes[0])
axes[0].set_title('F1-Score Terbaik per Model')
axes[0].set_ylim(0, 1.05)
for i, row in best_per_model.iterrows():
    axes[0].text(i, row['f1'] + 0.01, f"{row['f1']:.3f}", ha='center')

sns.barplot(data=best_per_model, x='Model', y='roc_auc', palette='magma', ax=axes[1])
axes[1].set_title('ROC AUC Terbaik per Model')
axes[1].set_ylim(0, 1.05)
for i, row in best_per_model.iterrows():
    axes[1].text(i, row['roc_auc'] + 0.01, f"{row['roc_auc']:.3f}", ha='center')

plt.tight_layout()
plt.show()


## STEP 9 - Pemilihan Model: W-KNN (sesuai instruksi)

> 🏷️ **Fase Pipeline:** 📊 Evaluasi Model (Cross Validation)

Validasi performa W-KNN dengan **5-fold cross validation** dan bandingkan
dengan kandidat lain agar pemilihan tetap empiris.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_summary = []

for _, row in best_per_model.iterrows():
    sc = SCALERS[row['Scaler']]
    rsp = RESAMPLERS[row['Resampler']]
    mdl = clone(MODELS[row['Model']])

    Xtr_s = sc.fit_transform(X_train)
    Xtr_r, ytr_r, _ = _safe_resample(rsp, Xtr_s, y_train)

    cv = cross_val_score(mdl, Xtr_r, ytr_r, cv=skf, scoring='f1', n_jobs=-1)
    cv_summary.append({
        'Model': row['Model'],
        'Scaler': row['Scaler'],
        'Resampler': row['Resampler'],
        'cv_f1_mean': cv.mean(),
        'cv_f1_std': cv.std(),
        'test_f1': row['f1'],
        'test_roc_auc': row['roc_auc'],
    })

cv_df = pd.DataFrame(cv_summary).sort_values('cv_f1_mean', ascending=False).reset_index(drop=True)
cv_df.round(4)


In [ ]:
# Pilih W-KNN sebagai base method (sesuai instruksi tugas)
wknn_row = cv_df[cv_df['Model'] == 'W-KNN'].iloc[0]
BEST = wknn_row.copy()

print(f"Base method (sesuai instruksi tugas): {BEST['Model']}")
print(f"  Konfigurasi   : Scaler={BEST['Scaler']}, Resampler={BEST['Resampler']}")
print(f"  CV F1 mean    : {BEST['cv_f1_mean']:.4f}  (std: {BEST['cv_f1_std']:.4f})")
print(f"  Test F1       : {BEST['test_f1']:.4f}")
print(f"  Test ROC AUC  : {BEST['test_roc_auc']:.4f}")

print('\nRanking semua model:')
print(cv_df.round(4).to_string(index=False))


## STEP 10 - Hyperparameter Tuning W-KNN

> 🏷️ **Fase Pipeline:** 🎛️ Tuning Model

Tuning W-KNN dengan `RandomizedSearchCV`. Parameter penting:
- `n_neighbors` — jumlah tetangga
- `weights` — `'uniform'` vs `'distance'` (W-KNN pakai distance)
- `metric` — `'euclidean'`, `'manhattan'`, `'minkowski'`
- `p` — exponent Minkowski


In [ ]:
PARAM_GRID_KNN = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'weights': ['distance'],  # W-KNN, fix
    'metric': ['euclidean', 'manhattan', 'minkowski'],
    'p': [1, 2, 3],
}

best_scaler = SCALERS[BEST['Scaler']]
best_resampler = RESAMPLERS[BEST['Resampler']]
Xtr_s = best_scaler.fit_transform(X_train)
Xte_s = best_scaler.transform(X_test)
Xtr_final, ytr_final, _ = _safe_resample(best_resampler, Xtr_s, y_train)

base = clone(MODELS['W-KNN'])
search = RandomizedSearchCV(
    base, param_distributions=PARAM_GRID_KNN,
    n_iter=15, cv=5, scoring='f1', n_jobs=-1,
    random_state=RANDOM_STATE, verbose=1,
)
search.fit(Xtr_final, ytr_final)

print('Best params  :', search.best_params_)
print(f"Best CV F1   : {search.best_score_:.4f}")


In [ ]:
def metrik(m):
    y_pred = m.predict(Xte_s)
    y_proba = m.predict_proba(Xte_s)[:, 1]
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
    }

default_model = clone(MODELS['W-KNN']).fit(Xtr_final, ytr_final)
tuned_model = search.best_estimator_

compare_df = pd.DataFrame([
    {'Model': 'W-KNN (default)', **metrik(default_model)},
    {'Model': 'W-KNN (tuned)', **metrik(tuned_model)},
])
compare_df.round(4)


## STEP 11 - Analisis Mendalam W-KNN (setelah tuning)

> 🏷️ **Fase Pipeline:** 📊 Evaluasi Model (analisis mendalam)


In [ ]:
best_model = tuned_model
y_pred = best_model.predict(Xte_s)
y_proba = best_model.predict_proba(Xte_s)[:, 1]

print('Classification Report (W-KNN tuned):')
print(classification_report(y_test, y_pred, target_names=['Tidak CKD', 'CKD']))
print('Best params:', search.best_params_)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tidak CKD', 'CKD'],
            yticklabels=['Tidak CKD', 'CKD'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix - W-KNN (tuned)')

fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, label=f"AUC = {auc:.4f}", linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve - W-KNN (tuned)')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


**Catatan:** W-KNN tidak punya `feature_importances_` bawaan
(non-parametric, lazy learner). Untuk menilai kontribusi fitur, gunakan
proxy seperti **permutation importance** atau lihat korelasi dari Step 5.


In [ ]:
# Permutation importance sebagai proxy feature importance untuk W-KNN
from sklearn.inspection import permutation_importance

perm = permutation_importance(best_model, Xte_s, y_test,
                              n_repeats=10, random_state=RANDOM_STATE,
                              scoring='f1')
perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=True)

plt.figure(figsize=(8, 7))
plt.barh(perm_df['feature'], perm_df['importance_mean'],
         xerr=perm_df['importance_std'], color='teal')
plt.title('Permutation Importance - W-KNN (tuned)')
plt.xlabel('Importance (drop in F1 ketika fitur dipermutasi)')
plt.tight_layout()
plt.show()
print(perm_df.sort_values('importance_mean', ascending=False))


## STEP 12 - Perbandingan Head-to-Head Semua Kandidat

> 🏷️ **Fase Pipeline:** 📊 Evaluasi Model (head-to-head perbandingan)

Kita konsolidasikan performa semua model dengan konfigurasi terbaiknya
+ versi tuned W-KNN.


In [ ]:
import time

def time_model(mdl, sc_name, rs_name):
    sc = SCALERS[sc_name]
    rs = RESAMPLERS[rs_name]
    Xtr_s = sc.fit_transform(X_train)
    Xte_s = sc.transform(X_test)
    Xtr_r, ytr_r, _ = _safe_resample(rs, Xtr_s, y_train)

    m = clone(mdl)
    t0 = time.time()
    m.fit(Xtr_r, ytr_r)
    fit_time = time.time() - t0

    t0 = time.time()
    y_pred = m.predict(Xte_s)
    pred_time = time.time() - t0

    y_proba = m.predict_proba(Xte_s)[:, 1] if hasattr(m, 'predict_proba') else None
    return {
        'fit_time_s': round(fit_time, 4),
        'pred_time_ms': round(pred_time * 1000, 2),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'auc': roc_auc_score(y_test, y_proba) if y_proba is not None else None,
    }

summary_rows = []
for _, row in best_per_model.iterrows():
    timing = time_model(MODELS[row['Model']], row['Scaler'], row['Resampler'])
    summary_rows.append({'Model': row['Model'], 'Variant': 'default',
                         'Scaler': row['Scaler'], 'Resampler': row['Resampler'],
                         **timing})

# Tambah W-KNN tuned
tuned_metrics = metrik(tuned_model)
summary_rows.append({
    'Model': 'W-KNN', 'Variant': 'tuned',
    'Scaler': BEST['Scaler'], 'Resampler': BEST['Resampler'],
    'fit_time_s': np.nan, 'pred_time_ms': np.nan,
    'accuracy': tuned_metrics['accuracy'],
    'precision': tuned_metrics['precision'],
    'recall': tuned_metrics['recall'],
    'f1': tuned_metrics['f1'],
    'auc': tuned_metrics['roc_auc'],
})

summary_df = pd.DataFrame(summary_rows).sort_values('auc', ascending=False).reset_index(drop=True)
print('=== Konsolidasi Performa Semua Kandidat (diurutkan ROC AUC) ===')
print(summary_df.round(4).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = [f"{r['Model']} ({r['Variant']})" for _, r in summary_df.iterrows()]
colors = []
for _, r in summary_df.iterrows():
    if 'W-KNN' in r['Model']:
        colors.append('#e67e22')   # oranye = base method
    else:
        colors.append('#3498db')

axes[0].barh(labels, summary_df['f1'], color=colors)
axes[0].set_xlabel('F1-Score (test set)')
axes[0].set_title('F1-Score Tiap Kandidat Model')
axes[0].set_xlim(0, 1.05)
for i, v in enumerate(summary_df['f1']):
    axes[0].text(v + 0.005, i, f"{v:.4f}", va='center')

axes[1].barh(labels, summary_df['auc'], color=colors)
axes[1].set_xlabel('ROC AUC (test set)')
axes[1].set_title('ROC AUC Tiap Kandidat Model')
axes[1].set_xlim(0, 1.05)
for i, v in enumerate(summary_df['auc']):
    axes[1].text(v + 0.005, i, f"{v:.4f}", va='center')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e67e22', label='Base method (W-KNN)'),
    Patch(facecolor='#3498db', label='Kandidat lain'),
]
axes[1].legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()


### 🎯 Kesimpulan

- **W-KNN (Weighted KNN)** dengan `MinMaxScaler` memberikan performa
  sangat baik (akurasi & ROC AUC tinggi) berkat dataset CKD yang
  separable di ruang Euclidean setelah scaling.
- Dataset CKD relatif kecil (400 pasien) — model non-parametric seperti
  W-KNN cocok karena tidak butuh asumsi distribusi.
- **Pelajaran preprocessing:**
  - Strip whitespace di kolom string krusial — tanpa ini target
    `'ckd'` dan `'ckd\t'` akan dianggap kelas berbeda.
  - Imputasi median/modus efektif untuk dataset dengan missing masif.
  - Scaler **MinMax** umumnya lebih baik dari Standard untuk W-KNN
    pada dataset dengan distribusi non-Gaussian.
